# **Presentation Quiz Generator**
This system is a Google Colab application that utilizes a 4-bit quantized 7-billion parameter local language model to generate multiple-choice questions from user-provided text. The output is programmatically formatted into a PowerPoint presentation file.

# Hardware Requirements
You must run this system on a hardware-accelerated environment. The system will fail if executed on a standard CPU instance.
*   Navigate to **Runtime > Change runtime type**.
*   Select the **T4 GPU** and press save.

# Execution Architecture
The code is intentionally separated into two distinct execution blocks to optimize memory allocation and reduce latency.
1. Run the first cell to install necessary dependencies and load the model weights into the GPU Video RAM. Do this only ONCE per session.
    - Note: Execution time is dependent on network speed for the initial model download.
2. The second cell handles user input, model inference, JSON parsing, and slide generation. You can execute this cell how many times necessary to get outputs.
    - It will ask the user to input 3 things: Quiz Title, Lesson Text, and Number of Questions.

# Output Specifications
The system automatically downloads two files to the local machine upon completion:

* Raw Text File (.txt): Contains the raw JSON output from the language model. This serves as a diagnostic log if the parsing fails.

* Presentation File (.pptx): The finalized quiz presentation.

# Presentation Formatting Logic
The generated PowerPoint file adheres to the following structural rules:

* The aspect ratio follows a 4:3 layout.
* Each question generated will have a single page, which also contains the four choices.
* Font sizes for questions and choices adjust based on total character count to prevent text truncation. Limits are 32pt for standard text, 26pt for text exceeding 350 characters, and 20pt for text exceeding 600 characters.
* The answers are grouped 10 per slide using a two-column layout. The left column holds items 1 through 5, and the right column holds items 6 through 10. The font is statically sized at 54pt.

# Troubleshooting
If the system returns an error stating the model did not return a valid format, inspect the downloaded text file. This indicates the language model failed to adhere to the strict JSON schema. Rerunning Cell 2 will typically resolve this due to the stochastic nature of the generation process.

---
# **Initialization Cell**
Run this cell only ONCE. Refer to Execution Architecture.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes python-pptx

import json
import math
import torch
import re
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from pptx import Presentation
from pptx.util import Inches, Pt
from google.colab import files

print("Status: Initializing the language model.")
model_id = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
quant_config = BitsAndBytesConfig(load_in_4bit=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=quant_config,
    torch_dtype=torch.float16
)
print("Status: Initialization complete. Proceed to the next cell.")

---
# **Generation Cell**
Run this cell how many times needed (or until limit is reached). Refer to Execution Architecture.

In [ ]:
# User Input
print("Status: Collecting user input.")
quiz_title = input("Enter the quiz title: ")
lesson_text = input("Enter the lesson text: ")
question_count_input = input("Enter the number of questions: ")
question_count = int(question_count_input)

# File Naming
safe_title = re.sub(r'[^a-z0-9_]', '', quiz_title.lower().replace(' ', '_'))
if not safe_title:
    safe_title = "quiz"
txt_filename = f"{safe_title}_response.txt"
pptx_filename = f"{safe_title}.pptx"

# Question Generation
print("Status: Processing text and generating questions.")
prompt = f"""Generate exactly {question_count} multiple choice questions based on the text below.
You must generate all {question_count} questions completely. Do not use placeholders, abbreviations, or comments like "Additional questions follow".
The answer field must only contain the single correct letter (A, B, C, or D).
Output strictly in this JSON format and provide no additional text:
{{
  "questions": [
    {{"question": "Question text here", "options": ["A. Option 1", "B. Option 2", "C. Option 3", "D. Option 4"], "answer": "A"}}
  ]
}}
Text: {lesson_text}"""

messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=4096, temperature=0.1, do_sample=True)
response_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

# Response Saving
print(f"Status: Saving raw model response to {txt_filename}.")
with open(txt_filename, "w", encoding="utf-8") as f:
    f.write(response_text)

# JSON Parsing
print("Status: Parsing generated JSON data.")
try:
    start_idx = min([i for i in [response_text.find('['), response_text.find('{')] if i != -1] or [-1])
    end_idx = max(response_text.rfind(']'), response_text.rfind('}')) + 1

    if start_idx != -1 and end_idx != 0:
        json_str = response_text[start_idx:end_idx]
    else:
        json_str = response_text

    parsed_data = json.loads(json_str)

    if isinstance(parsed_data, dict):
        quiz_data = parsed_data.get("questions", [])
    elif isinstance(parsed_data, list):
        quiz_data = parsed_data
    else:
        quiz_data = []
except json.JSONDecodeError:
    print("Status: Error parsing JSON output.")
    quiz_data = []

# PPTX Generation
print("Status: Generating the presentation slides.")
prs = Presentation()

title_slide_layout = prs.slide_layouts[0]
slide = prs.slides.add_slide(title_slide_layout)
title = slide.shapes.title
title.text = quiz_title

title_only_layout = prs.slide_layouts[5]

for i, q in enumerate(quiz_data):
    slide = prs.slides.add_slide(title_only_layout)
    shapes = slide.shapes

    title_shape = shapes.title
    title_shape.text = f"Question {i + 1}"

    left = Inches(1.0)
    top = Inches(1.5)
    width = Inches(8.0)
    height = Inches(5.0)

    txBox = slide.shapes.add_textbox(left, top, width, height)
    tf = txBox.text_frame
    tf.word_wrap = True

    q_text = q.get("question", "")
    options = q.get("options", [])

    total_chars = len(q_text) + sum(len(opt) for opt in options)

    if total_chars > 600:
        font_size = Pt(20)
    elif total_chars > 350:
        font_size = Pt(26)
    else:
        font_size = Pt(32)

    if len(tf.paragraphs) > 0:
        p = tf.paragraphs[0]
    else:
        p = tf.add_paragraph()

    p.text = q_text
    p.font.size = font_size

    p_blank = tf.add_paragraph()
    p_blank.text = ""
    p_blank.font.size = font_size

    for option in options:
        p = tf.add_paragraph()
        p.text = option
        p.font.size = font_size

if quiz_data:
    slide = prs.slides.add_slide(title_slide_layout)
    title = slide.shapes.title
    title.text = "Answer Key"

    answers_per_slide = 10
    total_answers = len(quiz_data)
    total_answer_slides = math.ceil(total_answers / answers_per_slide)

    for slide_idx in range(total_answer_slides):
        slide = prs.slides.add_slide(title_only_layout)
        shapes = slide.shapes
        title_shape = shapes.title
        title_shape.text = f"Answers (Page {slide_idx + 1})"

        start_idx = slide_idx * answers_per_slide
        end_idx = min(start_idx + answers_per_slide, total_answers)

        current_slide_answers = quiz_data[start_idx:end_idx]

        left_box = slide.shapes.add_textbox(Inches(0.5), Inches(1.5), Inches(4.5), Inches(5.0))
        left_tf = left_box.text_frame

        right_box = slide.shapes.add_textbox(Inches(5.5), Inches(1.5), Inches(4.5), Inches(5.0))
        right_tf = right_box.text_frame

        for idx, ans_data in enumerate(current_slide_answers):
            global_q_num = start_idx + idx + 1
            answer_text = str(ans_data.get('answer', ''))
            display_text = f"{global_q_num}. {answer_text}"

            target_tf = left_tf if idx < 5 else right_tf

            if len(target_tf.paragraphs) == 1 and target_tf.paragraphs[0].text == "":
                p = target_tf.paragraphs[0]
            else:
                p = target_tf.add_paragraph()

            p.text = display_text
            p.font.size = Pt(54)

print(f"Status: Saving presentation to {pptx_filename} and triggering downloads.")
prs.save(pptx_filename)
files.download(txt_filename)
files.download(pptx_filename)
print("Status: Process complete.")